# 11 深度學習 — 參考解答

松柏護理之家退伍軍人症群聚事件 PyTorch 深度學習練習的完整解答。

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
import pandas as pd
import numpy as np
import torch
from torch import nn
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

torch.manual_seed(42)
np.random.seed(42)

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)
df["severe_outcome"] = ((df["hospitalized"] == 1) | (df["outcome"] == "dead")).astype(int)

num_cols = ["age"]
cat_cols = ["sex", "smoking_history", "functional_status", "wing"]
bin_cols = [
    "floor", "comorbidity_chf", "comorbidity_dm", "comorbidity_cancer",
    "comorbidity_copd", "immunosuppressed", "shower_use", "hydrotherapy_use",
]

X_df = pd.get_dummies(df[num_cols + cat_cols + bin_cols], drop_first=True)
X_np = X_df.values.astype(np.float32)
scaler = StandardScaler()
X_np[:, 0] = scaler.fit_transform(X_np[:, 0:1]).ravel()

idx = np.arange(len(X_np))
np.random.shuffle(idx)
split = int(0.7 * len(idx))
train_idx, val_idx = idx[:split], idx[split:]
input_dim = X_df.shape[1]


def train_model(model, y_col, max_epochs=300, patience=15, lr=1e-3):
    """訓練 + 早停，回傳 train/val losses 和最佳模型。"""
    y_np_local = df[y_col].values.astype(np.float32)
    y_tr = torch.tensor(y_np_local[train_idx]).unsqueeze(1)
    y_va = torch.tensor(y_np_local[val_idx]).unsqueeze(1)
    X_tr = torch.tensor(X_np[train_idx])
    X_va = torch.tensor(X_np[val_idx])

    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    train_losses, val_losses = [], []
    best_val_loss, counter, best_epoch = float("inf"), 0, 0
    best_state = None

    for epoch in range(max_epochs):
        model.train()
        optimizer.zero_grad()
        loss = loss_fn(model(X_tr), y_tr)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

        model.eval()
        with torch.no_grad():
            vl = loss_fn(model(X_va), y_va).item()
        val_losses.append(vl)

        if vl < best_val_loss:
            best_val_loss = vl
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            best_epoch = epoch
            counter = 0
        else:
            counter += 1
        if counter >= patience:
            break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        auc_tr = roc_auc_score(y_tr.numpy(), torch.sigmoid(model(X_tr)).numpy())
        auc_va = roc_auc_score(y_va.numpy(), torch.sigmoid(model(X_va)).numpy())

    return train_losses, val_losses, best_epoch, auc_tr, auc_va

## 題目 1：改變架構（三層隱藏層）

In [ ]:
torch.manual_seed(42)

# 原始架構
model_2layer = nn.Sequential(
    nn.Linear(input_dim, 32), nn.ReLU(),
    nn.Linear(32, 16), nn.ReLU(),
    nn.Linear(16, 1),
)
n2 = sum(p.numel() for p in model_2layer.parameters())
tl2, vl2, be2, auc_tr2, auc_va2 = train_model(model_2layer, "infected")

torch.manual_seed(42)

# 三層隱藏層
model_3layer = nn.Sequential(
    nn.Linear(input_dim, 64), nn.ReLU(),
    nn.Linear(64, 32), nn.ReLU(),
    nn.Linear(32, 16), nn.ReLU(),
    nn.Linear(16, 1),
)
n3 = sum(p.numel() for p in model_3layer.parameters())
tl3, vl3, be3, auc_tr3, auc_va3 = train_model(model_3layer, "infected")

print("=== 架構比較 ===")
print(f"2 hidden layers: params={n2:,}, Train AUC={auc_tr2:.3f}, Val AUC={auc_va2:.3f}, gap={auc_tr2-auc_va2:.3f}")
print(f"3 hidden layers: params={n3:,}, Train AUC={auc_tr3:.3f}, Val AUC={auc_va3:.3f}, gap={auc_tr3-auc_va3:.3f}")
print(f"\n\u2192 更複雜的架構參數更多（{n3} vs {n2}），但 Val AUC 未必更好")
print("\u2192 Train-Val gap 越大 = 過擬合越嚴重")

## 題目 2：Task B — 預測重症

In [ ]:
torch.manual_seed(42)

model_b = nn.Sequential(
    nn.Linear(input_dim, 32), nn.ReLU(),
    nn.Linear(32, 16), nn.ReLU(),
    nn.Linear(16, 1),
)

tl_b, vl_b, be_b, auc_tr_b, auc_va_b = train_model(model_b, "severe_outcome")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(tl_b, label="Train Loss", color="#2c7fb8")
ax.plot(vl_b, label="Val Loss", color="#e34a33")
ax.axvline(x=be_b, color="gray", linestyle="--", alpha=0.5,
           label=f"Best epoch ({be_b})")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Learning Curve \u2014 Task B (severe_outcome)")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Task B: Train AUC={auc_tr_b:.3f}, Val AUC={auc_va_b:.3f}")
print(f"Task A: Train AUC={auc_tr2:.3f}, Val AUC={auc_va2:.3f}")
print(f"\n\u2192 Task B（重症）的正例比例更低（~24%），預測更困難")
print("\u2192 在小樣本 + 少正例的情況下，DL 表現通常不佳")

## 題目 3（挑戰題）：加入 Dropout

In [ ]:
torch.manual_seed(42)

# 含 Dropout
model_drop = nn.Sequential(
    nn.Linear(input_dim, 32), nn.ReLU(), nn.Dropout(0.3),
    nn.Linear(32, 16), nn.ReLU(), nn.Dropout(0.3),
    nn.Linear(16, 1),
)

tl_d, vl_d, be_d, auc_tr_d, auc_va_d = train_model(model_drop, "infected")

# 學習曲線比較
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(tl2, label="Train", color="#2c7fb8")
axes[0].plot(vl2, label="Val", color="#e34a33")
axes[0].set_title("No Dropout")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].plot(tl_d, label="Train", color="#2c7fb8")
axes[1].plot(vl_d, label="Val", color="#e34a33")
axes[1].set_title("With Dropout(0.3)")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()

plt.tight_layout()
plt.show()

print("=== Dropout 比較 ===")
print(f"No Dropout:   Train AUC={auc_tr2:.3f}, Val AUC={auc_va2:.3f}, gap={auc_tr2-auc_va2:.3f}")
print(f"Dropout(0.3): Train AUC={auc_tr_d:.3f}, Val AUC={auc_va_d:.3f}, gap={auc_tr_d-auc_va_d:.3f}")
print(f"\n\u2192 Dropout 會讓 train loss 較高（因為隨機關閉神經元）")
print("\u2192 但 Train-Val gap 通常較小 = 過擬合減少")
print("\u2192 在 280 筆資料上，Dropout 的效果有限，根本解法是增加資料量")

### 解讀

- **架構複雜度**：更多參數 ≠ 更好表現。280 筆資料只需要最簡單的架構
- **Task A vs B**：正例比例影響模型表現，少正例更需要適當的 loss 或 sampling 策略
- **Dropout**：是 DL 最常用的正則化技巧，但在極小樣本中效果有限
- **根本問題**：280 筆表格資料用 DL 不合理。DL 的優勢在於影像、文本、序列等非結構化資料
- **實務建議**：流病資料通常 n < 10,000 → 用 sklearn；n > 100,000 且有非結構化特徵 → 考慮 DL